# Calorimeter edges and vertex outliers

Explore particle-level cuts on the output of `macros/edge_outlier_study.C`, which writes one entry per reco
particle **before any cut**:
```
root -l -b -q 'macros/edge_outlier_study.C("<file or dir/*.hipo>","plots/edge_outlier_study/study.root",-1)'
```
* `pass_prod`: kept by the production cuts (`CutManager::filter_particles`).
* `pass_loose`: kept by the production cuts with **chi2pid and the v_z / v_T outlier cuts switched off**. Use it
  as the baseline when the cut under study is one of those three.
* `truepid`, `truep`, `trueE`: the matched MC truth (MC::RecMatch bank), −999 when unmatched and on real data.

Production values for reference: v_z ∈ (−40, 30) cm, v_T = √(v_x² + v_y²) < 20 cm, |χ²_pid| < 3 (pions:
momentum dependent), calorimeter edge min(lv, lw) ≥ 9 cm (e⁻) / 14 cm (γ) in PCAL and none for hadrons.

Each plot cell is a config block and one `overlay(...)` call. To try another cut, edit its selections dict.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import uproot

FILE = "plots/edge_outlier_study/study.root"

df = uproot.open(FILE)["particles"].arrays(library="pd")
print(f"{len(df):,} reco particles   pass_loose {df.pass_loose.mean():.1%}   pass_prod {df.pass_prod.mean():.1%}   "
      f"matched to truth {(df.truepid != -999).mean():.1%}")

In [ ]:
# dataviz reference palette: categorical slots in fixed order, chart surface and ink
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
plt.rcParams.update({"axes.facecolor": "#fcfcfb", "figure.facecolor": "#fcfcfb", "text.color": "#0b0b0b",
                     "axes.labelcolor": "#0b0b0b", "axes.edgecolor": "#c3c2b7", "xtick.color": "#52514e",
                     "ytick.color": "#52514e", "font.size": 9})


def calo_edge(d, layer="pcal"):
    """Distance (cm) to the nearest calorimeter edge, min(lv, lw) as in CutManager::caloEdges, in layer 'pcal',
    'ecin' or 'ecout'; NaN where that layer has no energy. 'first' takes the innermost layer that was hit."""
    if layer == "first":
        return calo_edge(d, "pcal").fillna(calo_edge(d, "ecin")).fillna(calo_edge(d, "ecout"))
    return np.minimum(d[f"{layer}_lv"], d[f"{layer}_lw"]).where(d[f"{layer}_e"] > 0)


def vz_window(d, lo, hi):
    return (d.vz > lo) & (d.vz < hi)


def vt_max(d, r):
    return np.hypot(d.vx, d.vy) < r


def width68(x):
    """Half the central 68% interval: a resolution width that ignores the tails."""
    if len(x) == 0:
        return np.nan
    lo, hi = np.percentile(x, [16, 84])
    return (hi - lo) / 2


def overlay(values, selections, bins, xlabel, density=False, logy=False):
    """One step histogram of `values` per named boolean mask (None = all), with N, mean and σ68 in the legend."""
    assert len(selections) <= len(COLORS), "too many selections for one plot: split them"
    fig, ax = plt.subplots(figsize=(12, 4.5), constrained_layout=True)
    for color, (label, sel) in zip(COLORS, selections.items()):
        v = values if sel is None else values[sel]
        ax.hist(v, bins=bins, histtype="step", lw=2, color=color, density=density,
                label=f"{label}   N={len(v):,}  mean={v.mean():.3g}  σ68={width68(v):.3g}")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("normalized" if density else "particles")
    if logy:
        ax.set_yscale("log")
    ax.grid(color="#e1e0d9", lw=0.5)
    ax.set_axisbelow(True)
    ax.legend(fontsize=8, frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1))
    plt.show()

## Hadron energy resolution vs. calorimeter edge cut

Matched particles that pass the production cuts; each edge cut is applied on top (production has none for
hadrons). `LAYER = "pcal"` measures the edge the way production does for e⁻/γ; `"first"` uses the innermost
layer that was hit, since neutral hadrons often leave nothing in PCAL. A particle with no hit in that layer fails
every edge cut.

In [ ]:
PIDS = [2112]            # reco pid(s)
LAYER = "first"          # 'pcal', 'ecin', 'ecout' or 'first'
EDGES = [5, 9, 14, 20]   # minimum distance to the edge (cm)
VAR = "E"                # 'E' or 'p', compared with 'true' + VAR
SAME_PID = True          # keep only particles whose matched truth has the reco pid

h = df[df.pass_prod & df.pid.isin(PIDS) & (df.truepid != -999)]
if SAME_PID:
    h = h[h.truepid == h.pid]
res = (h[VAR] - h["true" + VAR]) / h["true" + VAR]
edge = calo_edge(h, LAYER)

overlay(res, {"no edge cut": None, **{f"edge ≥ {d} cm": edge >= d for d in EDGES}},
        bins=np.linspace(-1, 1, 101), xlabel=f"({VAR} − true{VAR}) / true{VAR}", density=True)

## chi2pid after the v_z / v_T outlier cuts

Particles that pass `pass_loose`, so the production chi2pid cut has not truncated the distribution. The last
selection shows the particles the production outlier cuts remove. Set `VAR` and `BINS` to look at another
variable (e.g. `"vz"`, `"beta"`) under the same selections.

In [ ]:
PIDS = [211, -211]                 # reco pid(s)
VAR = "chi2"
BINS = np.linspace(-15, 15, 121)

c = df[df.pass_loose & df.pid.isin(PIDS)]
prod = vz_window(c, -40, 30) & vt_max(c, 20)

overlay(c[VAR], {
    "no v_z / v_T cut": None,
    "production: v_z ∈ (−40, 30) cm, v_T < 20 cm": prod,
    "v_z ∈ (−30, 25) cm, v_T < 20 cm": vz_window(c, -30, 25) & vt_max(c, 20),
    "v_z ∈ (−40, 30) cm, v_T < 15 cm": vz_window(c, -40, 30) & vt_max(c, 15),
    "removed by the production v_z / v_T cuts": ~prod,
}, bins=BINS, xlabel=VAR, logy=True)